In [1]:
# ================================================================
# 🧬 RESEARCHRADAR AI — COMPLETE COLAB PROJECT
# AI-Powered Scientific Research Intelligence Platform
# ================================================================

!pip -q install pandas numpy requests plotly scikit-learn networkx beautifulsoup4

# ================================================================
# IMPORTS
# ================================================================

import requests
import pandas as pd
import numpy as np
import re
import time
import warnings
import networkx as nx

import plotly.express as px
import plotly.graph_objects as go

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ================================================================
# CONFIGURATION
# ================================================================

PROJECT_NAME = "ResearchRadar AI"

display(Markdown("""
# 🧬 ResearchRadar AI

## AI-Powered Scientific Research Intelligence Platform

**Pipeline**

`PubMed → Data Engineering → NLP → Topic Modeling → Trend Detection → Research Intelligence → Visualization`

---
"""))

# ================================================================
# PUBMED API
# ================================================================

def search_pubmed(query, max_results=500):

    print(f" Searching PubMed for: {query}")

    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json",
        "sort": "date"
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    ids = data["esearchresult"]["idlist"]

    print(f" Found {len(ids)} papers")

    return ids


# ================================================================
# FETCH PUBMED DETAILS
# ================================================================

def fetch_pubmed(ids):

    if not ids:
        return pd.DataFrame()

    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"

    records = []

    # Fetch in batches
    for start in range(0, len(ids), 100):

        batch = ids[start:start+100]

        params = {
            "db": "pubmed",
            "id": ",".join(batch),
            "retmode": "json"
        }

        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        data = response.json()

        result = data.get("result", {})

        for pid in batch:

            item = result.get(pid)

            if not item:
                continue

            title = item.get(
                "title",
                "Unknown title"
            )

            pubdate = item.get(
                "pubdate",
                ""
            )

            try:
                year = int(
                    re.search(
                        r"\d{4}",
                        pubdate
                    ).group()
                )
            except:
                year = np.nan

            authors = item.get(
                "authors",
                []
            )

            author_names = [
                a.get(
                    "name",
                    ""
                )
                for a in authors
            ]

            journal = item.get(
                "fulljournalname",
                "Unknown"
            )

            records.append({

                "pmid": pid,

                "title": title,

                "year": year,

                "authors": "; ".join(
                    author_names
                ),

                "first_author": (
                    author_names[0]
                    if author_names
                    else "Unknown"
                ),

                "journal": journal,

                "pubdate": pubdate

            })

        time.sleep(0.15)

    return pd.DataFrame(records)


# ================================================================
# ABSTRACT FETCHER
# ================================================================

def fetch_abstracts(ids):

    if not ids:
        return {}

    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    abstracts = {}

    for start in range(0, len(ids), 100):

        batch = ids[start:start+100]

        params = {
            "db": "pubmed",
            "id": ",".join(batch),
            "rettype": "abstract",
            "retmode": "text"
        }

        try:

            response = requests.get(
                url,
                params=params,
                timeout=40
            )

            text = response.text

            # Split approximately by PMID
            chunks = re.split(
                r"\nPMID-\s+",
                text
            )

            for chunk in chunks:

                match = re.search(
                    r"^(\d+)",
                    chunk
                )

                if not match:
                    continue

                pmid = match.group(1)

                abs_match = re.search(
                    r"AB\s+-\s+(.*?)(?=\n[A-Z]{2}\s+-|\n[A-Z]{2}\s{2}-|\Z)",
                    chunk,
                    re.S
                )

                if abs_match:

                    abstract = re.sub(
                        r"\s+",
                        " ",
                        abs_match.group(1)
                    ).strip()

                    abstracts[pmid] = abstract

        except Exception:
            pass

        time.sleep(0.2)

    return abstracts


# ================================================================
# DEMO DATA FALLBACK
# ================================================================

def create_demo_data(n=1200):

    np.random.seed(42)

    topics = {

        "Protein Language Models": [
            "protein language model",
            "protein embeddings",
            "transformer",
            "protein prediction"
        ],

        "AI Drug Discovery": [
            "drug discovery",
            "molecular generation",
            "drug design",
            "deep learning"
        ],

        "Variant Pathogenicity": [
            "variant pathogenicity",
            "genomic variant",
            "disease variant",
            "variant prediction"
        ],

        "Single Cell AI": [
            "single cell",
            "single cell transcriptomics",
            "cell embeddings",
            "single cell analysis"
        ],

        "Foundation Models": [
            "foundation model",
            "large language model",
            "multimodal model",
            "generative AI"
        ],

        "Multi Omics": [
            "multi omics",
            "genomics",
            "proteomics",
            "transcriptomics"
        ],

        "Gene Regulation": [
            "gene regulation",
            "gene expression",
            "epigenomics",
            "regulatory elements"
        ],

        "Medical AI": [
            "medical AI",
            "clinical AI",
            "healthcare",
            "medical imaging"
        ],

        "CRISPR AI": [
            "CRISPR",
            "gene editing",
            "CRISPR prediction",
            "machine learning"
        ],

        "Biomedical NLP": [
            "biomedical NLP",
            "text mining",
            "clinical language model",
            "medical language"
        ]
    }

    authors = [
        "A Sharma",
        "R Kumar",
        "M Singh",
        "S Patel",
        "A Verma",
        "N Gupta",
        "P Rao",
        "K Mehta",
        "D Khan",
        "S Iyer"
    ]

    journals = [
        "Nature Biotechnology",
        "Nature Methods",
        "Bioinformatics",
        "Genome Biology",
        "Scientific Reports",
        "PLOS Computational Biology",
        "Briefings in Bioinformatics"
    ]

    rows = []

    topic_list = list(topics.keys())

    for i in range(n):

        topic = np.random.choice(
            topic_list
        )

        keywords = topics[topic]

        year = np.random.choice(
            range(2019, 2027),
            p=[
                .05, .06, .08, .10,
                .12, .17, .20, .22
            ]
        )

        title = (
            f"{keywords[0].title()} "
            f"for Computational Biomedical Research"
        )

        abstract = (
            f"This study investigates "
            f"{keywords[0]} using "
            f"machine learning and "
            f"computational biology. "
            f"The research explores "
            f"{keywords[1]} and "
            f"{keywords[2]} "
            f"for biomedical applications."
        )

        citations = max(
            0,
            int(
                np.random.normal(
                    20 + (year - 2019) * 8,
                    25
                )
            )
        )

        rows.append({

            "pmid": f"DEMO{i}",

            "title": title,

            "abstract": abstract,

            "year": year,

            "first_author":
                np.random.choice(authors),

            "journal":
                np.random.choice(journals),

            "citations":
                citations,

            "topic":
                topic

        })

    return pd.DataFrame(rows)


# ================================================================
# GET DATA
# ================================================================

QUERY = "artificial intelligence AND genomics"

try:

    ids = search_pubmed(
        QUERY,
        max_results=500
    )

    df = fetch_pubmed(ids)

    if not df.empty:

        abstracts = fetch_abstracts(
            ids
        )

        df["abstract"] = df[
            "pmid"
        ].map(
            abstracts
        ).fillna("")

        df["citations"] = 0

        df["topic"] = "AI + Genomics"

        REAL_DATA = True

    else:

        raise Exception()

except Exception as e:

    print(
        "⚠️ PubMed retrieval unavailable."
    )

    print(
        "Using demonstration dataset."
    )

    df = create_demo_data()

    REAL_DATA = False


# ================================================================
# CLEAN DATA
# ================================================================

df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
)

df = df.dropna(
    subset=["year"]
)

df["year"] = df[
    "year"
].astype(int)

df["title"] = df[
    "title"
].fillna("")

df["abstract"] = df[
    "abstract"
].fillna("")

df["text"] = (
    df["title"] + " " +
    df["abstract"]
)

# Remove impossible years
df = df[
    (df["year"] >= 2000) &
    (df["year"] <= 2026)
]

df = df.drop_duplicates(
    subset=["title"]
)

# ================================================================
# DATASET SUMMARY
# ================================================================

display(
    Markdown(
        "## 📊 Dataset Intelligence"
    )
)

summary = pd.DataFrame({

    "Metric": [
        "Papers",
        "Years",
        "Authors",
        "Journals",
        "Average Paper Length"
    ],

    "Value": [

        len(df),

        f"{df.year.min()}–{df.year.max()}",

        df["first_author"].nunique(),

        df["journal"].nunique(),

        int(
            df["text"]
            .str.len()
            .mean()
        )
    ]
})

display(summary)

# ================================================================
# 1. PUBLICATION TREND
# ================================================================

display(
    Markdown(
        "## 📈 1. Publication Trend"
    )
)

yearly = (
    df.groupby("year")
      .size()
      .reset_index(
          name="papers"
      )
)

fig = px.area(

    yearly,

    x="year",

    y="papers",

    markers=True,

    title="Research Publication Activity",

    template="plotly_dark"
)

fig.update_layout(
    height=500
)

fig.show()

# ================================================================
# 2. NLP KEYWORDS
# ================================================================

display(
    Markdown(
        "##  2. Research Keyword Intelligence"
    )
)

vectorizer = TfidfVectorizer(

    stop_words="english",

    ngram_range=(1,3),

    min_df=2,

    max_features=5000
)

X = vectorizer.fit_transform(
    df["text"]
)

scores = np.asarray(
    X.mean(
        axis=0
    )
).ravel()

terms = (
    vectorizer
    .get_feature_names_out()
)

keyword_df = pd.DataFrame({

    "keyword": terms,

    "score": scores

})

keyword_df = (
    keyword_df
    .sort_values(
        "score",
        ascending=False
    )
    .head(25)
)

fig = px.bar(

    keyword_df.sort_values(
        "score"
    ),

    x="score",

    y="keyword",

    orientation="h",

    title="Top Research Keywords",

    template="plotly_dark"
)

fig.update_layout(
    height=700
)

fig.show()

# ================================================================
# 3. TOP AUTHORS
# ================================================================

display(
    Markdown(
        "##  3. Researcher Intelligence"
    )
)

authors_df = (
    df["first_author"]
    .value_counts()
    .head(15)
    .reset_index()
)

authors_df.columns = [
    "author",
    "papers"
]

fig = px.bar(

    authors_df.sort_values(
        "papers"
    ),

    x="papers",

    y="author",

    orientation="h",

    text="papers",

    title="Most Active Researchers",

    template="plotly_dark"
)

fig.update_layout(
    height=600
)

fig.show()

# ================================================================
# 4. JOURNAL ANALYSIS
# ================================================================

display(
    Markdown(
        "##  4. Journal Intelligence"
    )
)

journal_df = (
    df["journal"]
    .value_counts()
    .head(15)
    .reset_index()
)

journal_df.columns = [
    "journal",
    "papers"
]

fig = px.bar(

    journal_df.sort_values(
        "papers"
    ),

    x="papers",

    y="journal",

    orientation="h",

    title="Research Distribution by Journal",

    template="plotly_dark"
)

fig.update_layout(
    height=650
)

fig.show()

# ================================================================
# 5. TOPIC CLUSTERING
# ================================================================

display(
    Markdown(
        "##  5. Machine Learning Topic Discovery"
    )
)

K = min(
    8,
    max(
        3,
        len(df)//50
    )
)

cluster_vectorizer = TfidfVectorizer(

    stop_words="english",

    ngram_range=(1,2),

    max_features=4000
)

cluster_X = (
    cluster_vectorizer
    .fit_transform(
        df["text"]
    )
)

kmeans = KMeans(

    n_clusters=K,

    random_state=42,

    n_init=10
)

df["cluster"] = (
    kmeans
    .fit_predict(
        cluster_X
    )
)

cluster_terms = (
    cluster_vectorizer
    .get_feature_names_out()
)

cluster_names = {}

for cluster in range(K):

    center = (
        kmeans
        .cluster_centers_[cluster]
    )

    top_indices = (
        center
        .argsort()[-5:][::-1]
    )

    words = [
        cluster_terms[i]
        for i in top_indices
    ]

    cluster_names[
        cluster
    ] = " • ".join(
        words[:3]
    )

df["cluster_name"] = (
    df["cluster"]
    .map(cluster_names)
)

cluster_df = (
    df["cluster_name"]
    .value_counts()
    .reset_index()
)

cluster_df.columns = [
    "research_cluster",
    "papers"
]

display(cluster_df)

fig = px.bar(

    cluster_df.sort_values(
        "papers"
    ),

    x="papers",

    y="research_cluster",

    orientation="h",

    text="papers",

    title="Automatically Discovered Research Themes",

    template="plotly_dark"
)

fig.update_layout(
    height=650
)

fig.show()

# ================================================================
# 6. TOPIC EVOLUTION
# ================================================================

display(
    Markdown(
        "##  6. Research Theme Evolution"
    )
)

topic_year = (
    df.groupby(
        [
            "year",
            "cluster_name"
        ]
    )
    .size()
    .reset_index(
        name="papers"
    )
)

top_clusters = (
    df["cluster_name"]
    .value_counts()
    .head(6)
    .index
)

topic_year = topic_year[
    topic_year[
        "cluster_name"
    ].isin(
        top_clusters
    )
]

fig = px.line(

    topic_year,

    x="year",

    y="papers",

    color="cluster_name",

    markers=True,

    title="Evolution of Research Themes",

    template="plotly_dark"
)

fig.update_layout(
    height=650
)

fig.show()

# ================================================================
# 7. EMERGING TOPIC DETECTOR
# ================================================================

display(
    Markdown(
        "##  7. Emerging Research Detector"
    )
)

latest_year = df["year"].max()

previous_year = latest_year - 1

latest = (
    df[
        df["year"] == latest_year
    ]["cluster_name"]
    .value_counts()
)

previous = (
    df[
        df["year"] == previous_year
    ]["cluster_name"]
    .value_counts()
)

all_clusters = set(
    latest.index
).union(
    previous.index
)

emerging = []

for cluster in all_clusters:

    current_count = latest.get(
        cluster,
        0
    )

    previous_count = previous.get(
        cluster,
        0
    )

    growth = (
        current_count -
        previous_count
    ) / max(
        previous_count,
        1
    )

    emerging.append({

        "research_area":
            cluster,

        "current_papers":
            current_count,

        "previous_papers":
            previous_count,

        "growth":
            growth
    })

emerging_df = pd.DataFrame(
    emerging
)

if len(emerging_df) > 1:

    scaler = MinMaxScaler()

    emerging_df[
        "momentum_score"
    ] = (
        scaler
        .fit_transform(
            emerging_df[
                ["growth"]
            ]
        )
        .flatten()
        * 100
    )

else:

    emerging_df[
        "momentum_score"
    ] = 50

emerging_df = (
    emerging_df
    .sort_values(
        "momentum_score",
        ascending=False
    )
)

display(
    emerging_df
    .head(10)
    .style.format({
        "growth": "{:.1%}",
        "momentum_score": "{:.1f}"
    })
)

fig = px.bar(

    emerging_df.head(10),

    x="momentum_score",

    y="research_area",

    orientation="h",

    text="momentum_score",

    title="Research Momentum Index",

    template="plotly_dark"
)

fig.update_layout(
    height=600
)

fig.show()

# ================================================================
# 8. RESEARCH HEATMAP
# ================================================================

display(
    Markdown(
        "##  8. Research Activity Heatmap"
    )
)

heatmap = (
    df.groupby(
        [
            "year",
            "cluster_name"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

fig = px.imshow(

    heatmap.T,

    aspect="auto",

    labels={
        "x": "Year",
        "y": "Research Theme",
        "color": "Papers"
    },

    title="Research Activity by Year and Theme",

    template="plotly_dark"
)

fig.update_layout(
    height=700
)

fig.show()

# ================================================================
# 9. ANOMALY DETECTION
# ================================================================

display(
    Markdown(
        "##  9. Research Activity Anomaly Detection"
    )
)

annual = (
    df.groupby("year")
      .size()
      .reset_index(
          name="papers"
      )
)

if len(annual) >= 5:

    anomaly_model = (
        IsolationForest(
            contamination=0.20,
            random_state=42
        )
    )

    annual["prediction"] = (
        anomaly_model
        .fit_predict(
            annual[["papers"]]
        )
    )

    annual["status"] = np.where(
        annual["prediction"] == -1,
        "Unusual",
        "Normal"
    )

else:

    annual["status"] = "Normal"

fig = px.bar(

    annual,

    x="year",

    y="papers",

    color="status",

    text="papers",

    title="Unusual Publication Activity",

    template="plotly_dark"
)

fig.update_layout(
    height=500
)

fig.show()

# ================================================================
# 10. RESEARCH NETWORK
# ================================================================

display(
    Markdown(
        "## 🕸️ 10. Research Knowledge Network"
    )
)

G = nx.Graph()

# Use top research clusters
network_clusters = (
    df["cluster_name"]
    .value_counts()
    .head(8)
    .index
)

for cluster in network_clusters:

    G.add_node(
        cluster,
        node_type="cluster"
    )

    cluster_rows = df[
        df["cluster_name"] == cluster
    ]

    # Extract important words from cluster
    combined_text = " ".join(
        cluster_rows["text"]
        .head(100)
    )

    words = re.findall(
        r"\b[a-zA-Z]{4,}\b",
        combined_text.lower()
    )

    stopwords = {
        "this", "that", "with",
        "from", "using", "study",
        "research", "analysis",
        "model", "models",
        "data", "based"
    }

    counts = Counter(
        w for w in words
        if w not in stopwords
    )

    for word, count in counts.most_common(5):

        G.add_node(
            word,
            node_type="keyword"
        )

        G.add_edge(
            cluster,
            word,
            weight=count
        )

pos = nx.spring_layout(
    G,
    seed=42,
    k=0.8
)

edge_x = []
edge_y = []

for source, target in G.edges():

    x0, y0 = pos[source]
    x1, y1 = pos[target]

    edge_x.extend(
        [x0, x1, None]
    )

    edge_y.extend(
        [y0, y1, None]
    )

edge_trace = go.Scatter(

    x=edge_x,

    y=edge_y,

    mode="lines",

    line=dict(
        width=1
    ),

    hoverinfo="none"
)

node_x = []
node_y = []
node_text = []
node_size = []

for node in G.nodes():

    x, y = pos[node]

    node_x.append(x)
    node_y.append(y)

    node_text.append(node)

    degree = G.degree(node)

    node_size.append(
        15 + degree * 4
    )

node_trace = go.Scatter(

    x=node_x,

    y=node_y,

    mode="markers+text",

    text=node_text,

    textposition="top center",

    marker=dict(
        size=node_size
    ),

    hoverinfo="text"
)

fig = go.Figure(
    data=[
        edge_trace,
        node_trace
    ]
)

fig.update_layout(

    title="Research Theme ↔ Keyword Network",

    template="plotly_dark",

    height=750,

    showlegend=False,

    xaxis=dict(
        showgrid=False,
        showticklabels=False,
        zeroline=False
    ),

    yaxis=dict(
        showgrid=False,
        showticklabels=False,
        zeroline=False
    )
)

fig.show()

# ================================================================
# 11. PAPER EXPLORER
# ================================================================

display(
    Markdown(
        "##  11. Paper Explorer"
    )
)

paper_view = df[
    [
        "pmid",
        "title",
        "year",
        "first_author",
        "journal",
        "cluster_name"
    ]
].copy()

display(
    paper_view
    .sort_values(
        "year",
        ascending=False
    )
    .head(50)
)

# ================================================================
# 12. AI-STYLE RESEARCH SUMMARY
# ================================================================

display(
    Markdown(
        "##  ResearchRadar Intelligence Summary"
    )
)

top_theme = (
    df["cluster_name"]
    .value_counts()
    .idxmax()
)

top_author = (
    df["first_author"]
    .value_counts()
    .idxmax()
)

top_journal = (
    df["journal"]
    .value_counts()
    .idxmax()
)

display(
    Markdown(
        f"""
###  Dataset Findings

**Papers analyzed:** `{len(df):,}`

**Research period:** `{df.year.min()} – {df.year.max()}`

**Largest discovered research theme:**
### `{top_theme}`

**Most active researcher in the dataset:**
### `{top_author}`

**Most represented journal:**
### `{top_journal}`

### 🚀 Emerging Research

The emerging-topic engine compares recent research activity
with the previous period and calculates a project-defined
**Research Momentum Index**.

###  Important

These findings describe the analyzed dataset. They should
not automatically be interpreted as representing the entire
scientific literature.

The momentum index is an analytical metric created by this
project rather than an established scientific ranking.
"""
    )
)

# ================================================================
# 13. EXPORT
# ================================================================

df.to_csv(
    "researchradar_processed.csv",
    index=False
)

emerging_df.to_csv(
    "researchradar_emerging_topics.csv",
    index=False
)

cluster_df.to_csv(
    "researchradar_clusters.csv",
    index=False
)

print("\n" + "="*60)
print(" RESEARCHRADAR AI COMPLETE")
print("="*60)

print(
    f"""
 Papers analyzed: {len(df):,}
 Research themes: {df['cluster_name'].nunique()}

Generated files:

✓ researchradar_processed.csv
✓ researchradar_emerging_topics.csv
✓ researchradar_clusters.csv
"""
)


# 🧬 ResearchRadar AI

## AI-Powered Scientific Research Intelligence Platform

**Pipeline**

`PubMed → Data Engineering → NLP → Topic Modeling → Trend Detection → Research Intelligence → Visualization`

---


 Searching PubMed for: artificial intelligence AND genomics
 Found 500 papers


## 📊 Dataset Intelligence

,Metric,Value
0,Papers,498
1,Years,2025–2026
2,Authors,471
3,Journals,300
4,Average Paper Length,110


## 📈 1. Publication Trend

...

##  11. Paper Explorer

,pmid,title,year,first_author,journal,cluster_name
499,42584423,Livestock Multi-Omics Integration: A Systemati...,2026,Wen J,"Advanced science (Weinheim, Baden-Wurttemberg,...",omics • multi • multi omics
0,42779146,Guiding the Application of Immunotherapy in No...,2026,Sun Y,Thoracic cancer,cancer • breast • breast cancer
483,42589315,Systematic Benchmarking of DNA Sequence Encodi...,2026,Jin H,International journal of molecular sciences,deep • deep learning • ai
482,42589412,Artificial Intelligence and Genomic Data Analy...,2026,Blaga AM,International journal of molecular sciences,intelligence • artificial intelligence • artif...
481,42589596,Distinct Modulation of High- and Low-Frequency...,2026,Bieletzki S,International journal of molecular sciences,deep • deep learning • ai
480,42589638,A Compact Multipartite Mitogenome of Clausena ...,2026,Bi C,International journal of molecular sciences,deep • deep learning • ai
479,42589691,Hepatic SIRT6 Deficiency Accelerates Female-Sp...,2026,Liu Y,International journal of molecular sciences,deep • deep learning • ai
478,42589990,Early Precise Prediction and Severe Risk Strat...,2026,Xu W,Journal of clinical medicine,intelligence • artificial intelligence • artif...
477,42590009,The Role of Muscle Biopsy in the Era of Modern...,2026,Sadeh M,Journal of clinical medicine,genomic • prediction • model
476,42590162,Artificial Intelligence in Obstetrics: Current...,2026,Many I,Journal of clinical medicine,intelligence • artificial intelligence • artif...


##  ResearchRadar Intelligence Summary


###  Dataset Findings

**Papers analyzed:** `498`

**Research period:** `2025 – 2026`

**Largest discovered research theme:**  
### `deep • deep learning • ai`

**Most active researcher in the dataset:**  
### `Wang Y`

**Most represented journal:**  
### `Bioinformatics (Oxford, England)`

### 🚀 Emerging Research

The emerging-topic engine compares recent research activity
with the previous period and calculates a project-defined
**Research Momentum Index**.

###  Important

These findings describe the analyzed dataset. They should
not automatically be interpreted as representing the entire
scientific literature.

The momentum index is an analytical metric created by this
project rather than an established scientific ranking.



 RESEARCHRADAR AI COMPLETE

 Papers analyzed: 498
 Research themes: 8

Generated files:

✓ researchradar_processed.csv
✓ researchradar_emerging_topics.csv
✓ researchradar_clusters.csv

